# Data Preprocessing — Video Game Sales

Notebook ini berisi tahapan preprocessing data berdasarkan temuan dari EDA.

| Tahap | Keterangan |
|-------|------------|
| 1 | Import Library & Load Data |
| 2 | Kondisi Data Awal |
| 3 | Menghapus Missing Value |
| 4 | Menghapus Data Duplikat |
| 5 | Mengatasi Data Tidak Konsisten |
| 6 | Mengubah Format Data |
| 7 | Feature Engineering |
| 8 | Filtering Data Tidak Relevan |
| 9 | Hasil Akhir & Export |

## 1. Import Library & Load Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset mentah
df = pd.read_csv("../data/raw/video_game_sales.csv")
print("Dataset berhasil dimuat.")
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")

Dataset berhasil dimuat.
Jumlah baris : 16,598
Jumlah kolom : 11


## 2. Kondisi Data Awal

Menampilkan kondisi dataset sebelum preprocessing sebagai **baseline** perbandingan.

In [6]:
print("=" * 50)
print("KONDISI DATA AWAL")
print("=" * 50)
print(f"Jumlah baris    : {df.shape[0]:,}")
print(f"Jumlah kolom    : {df.shape[1]}")
print(f"Jumlah duplikat : {df.duplicated().sum()}")
print()
print("Missing Value per Kolom:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah Missing'] > 0])
print()
print("Tipe Data:")
print(df.dtypes)

KONDISI DATA AWAL
Jumlah baris    : 16,598
Jumlah kolom    : 11
Jumlah duplikat : 0

Missing Value per Kolom:
           Jumlah Missing  Persentase (%)
Year                  271            1.63
Publisher              58            0.35

Tipe Data:
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object


**Temuan dari EDA:**
- Kolom `Year` memiliki **271 missing value** (1.63%) — bertipe float, seharusnya integer
- Kolom `Publisher` memiliki **58 missing value** (0.35%)
- Perlu dicek adanya duplikat
- Perlu dicek konsistensi nilai `Year` (ada tahun yang tidak wajar)

## 3. Menghapus Missing Value

Berdasarkan EDA, kolom yang memiliki missing value adalah `Year` (271 baris) dan `Publisher` (58 baris).
Karena jumlahnya kecil dibanding total data (16.598 baris), kita hapus baris tersebut.

In [5]:
df_clean = df.copy()

sebelum = len(df_clean)

# Hapus baris yang Year atau Publisher kosong
df_clean = df_clean.dropna(subset=['Year', 'Publisher'])

sesudah = len(df_clean)
terhapus = sebelum - sesudah

print("Missing value berhasil dihapus.")
print(f"Baris sebelum : {sebelum:,}")
print(f"Baris terhapus: {terhapus:,}")
print(f"Baris sesudah : {sesudah:,}")
print()
print("Sisa missing value:")
print(df_clean.isnull().sum())

Missing value berhasil dihapus.
Baris sebelum : 16,598
Baris terhapus: 307
Baris sesudah : 16,291

Sisa missing value:
Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


## 4. Menghapus Data Duplikat

Mengecek dan menghapus baris yang benar-benar identik di semua kolom.

In [7]:
sebelum = len(df_clean)
jumlah_duplikat = df_clean.duplicated().sum()

print(f"Jumlah duplikat ditemukan: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"{jumlah_duplikat} duplikat berhasil dihapus.")
else:
    print("Tidak ada duplikat. Data tetap utuh.")

print(f"Baris sebelum : {sebelum:,}")
print(f"Baris sesudah : {len(df_clean):,}")

Jumlah duplikat ditemukan: 0
Tidak ada duplikat. Data tetap utuh.
Baris sebelum : 16,291
Baris sesudah : 16,291


## 5. Mengatasi Data Tidak Konsisten

Mengecek nilai-nilai yang tidak wajar atau tidak konsisten di tiap kolom.

In [8]:
# ── 5a. Cek nilai Year yang tidak wajar ──────────────────────────
print("Nilai unik Year (luar rentang 1980–2016):")
tahun_aneh = df_clean[~df_clean['Year'].between(1980, 2016)]['Year'].unique()
print(tahun_aneh if len(tahun_aneh) > 0 else "Tidak ada.")

# Hapus baris dengan tahun di luar rentang wajar
sebelum = len(df_clean)
df_clean = df_clean[df_clean['Year'].between(1980, 2016)]
print(f"\nBaris terhapus karena Year tidak wajar: {sebelum - len(df_clean)}")

Nilai unik Year (luar rentang 1980–2016):
[2020. 2017.]

Baris terhapus karena Year tidak wajar: 4


In [9]:
# ── 5b. Cek konsistensi kolom teks (strip whitespace & title case) ─
kolom_teks = ['Name', 'Platform', 'Genre', 'Publisher']

for col in kolom_teks:
    df_clean[col] = df_clean[col].str.strip()          # hapus spasi di awal/akhir
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)  # hapus spasi ganda

print("Whitespace dan spasi ganda pada kolom teks berhasil dibersihkan.")
print(f"Kolom yang dibersihkan: {kolom_teks}")

Whitespace dan spasi ganda pada kolom teks berhasil dibersihkan.
Kolom yang dibersihkan: ['Name', 'Platform', 'Genre', 'Publisher']


In [10]:
# ── 5c. Cek nilai Global_Sales vs penjumlahan regional ─────────────
# Global_Sales seharusnya mendekati NA + EU + JP + Other
df_clean['_sales_check'] = (df_clean['NA_Sales'] + df_clean['EU_Sales'] +
                             df_clean['JP_Sales'] + df_clean['Other_Sales']).round(2)
df_clean['_selisih'] = (df_clean['Global_Sales'] - df_clean['_sales_check']).abs()

# Toleransi selisih pembulatan: <= 0.05
tidak_konsisten = df_clean[df_clean['_selisih'] > 0.05]
print(f"Baris dengan Global_Sales tidak konsisten (selisih > 0.05): {len(tidak_konsisten)}")

if len(tidak_konsisten) > 0:
    print("Contoh data tidak konsisten:")
    print(tidak_konsisten[['Name', 'NA_Sales', 'EU_Sales', 'JP_Sales',
                             'Other_Sales', 'Global_Sales', '_selisih']].head())

# Hapus kolom bantu
df_clean = df_clean.drop(columns=['_sales_check', '_selisih'])
print("\nPengecekan konsistensi Global_Sales selesai.")

Baris dengan Global_Sales tidak konsisten (selisih > 0.05): 0

Pengecekan konsistensi Global_Sales selesai.


## 6. Mengubah Format Data

Mengubah tipe data kolom agar sesuai dengan nilainya.

In [12]:
# ── 6a. Ubah Year dari float ke integer ───────────────────────────
print("Tipe data sebelum:")
print(df_clean[['Year']].dtypes)

df_clean['Year'] = df_clean['Year'].astype(int)

print("\nTipe data sesudah:")
print(df_clean[['Year']].dtypes)
print("\nKolom Year berhasil diubah dari float64 → int64.")

Tipe data sebelum:
Year    int64
dtype: object

Tipe data sesudah:
Year    int64
dtype: object

Kolom Year berhasil diubah dari float64 → int64.


In [14]:
# ── 6b. Pastikan kolom sales bertipe float dan tidak negatif ───────
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

for col in sales_cols:
    df_clean[col] = df_clean[col].astype(float)
    # Ganti nilai negatif (jika ada) dengan 0
    negatif = (df_clean[col] < 0).sum()
    if negatif > 0:
        df_clean[col] = df_clean[col].clip(lower=0)
        print(f'   {col}: {negatif} nilai negatif diganti 0')

print("Semua kolom sales sudah bertipe float dan tidak negatif.")

Semua kolom sales sudah bertipe float dan tidak negatif.
